# Stage 4: LoRA + Inference Control (CHAIR)

This notebook runs **inference-time grounding control on top of the Stage 2 LoRA adapter** and reports CHAIR metrics.

## Goal
- Load LLaVA base + Stage 2 LoRA adapter.
- Run CHAIR with standard greedy decoding (LoRA-only baseline).
- Run CHAIR with inference control enabled (LoRA + control).
- Save Stage 4 results for final comparison.

## Required inputs
- `results/final_hallucination_heads.json`
- `cache/screening_state.pkl`
- `cache/selected_imgs.json`
- `results/stage2_lora_adapter` (or your tagged adapter directory)
- COCO data under `coco/`

In [ ]:
# Optional installs for fresh runtimes (uncomment if needed)
# !pip -q install -U transformers peft bitsandbytes accelerate pycocotools spacy
# !python -m spacy download en_core_web_sm

In [ ]:
import os
import json
import random
import pickle
from pathlib import Path
from collections import defaultdict

import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from pycocotools.coco import COCO

import spacy
from transformers import AutoProcessor, LlavaForConditionalGeneration
from peft import PeftModel

nlp = spacy.load('en_core_web_sm')

# If running in Colab, mount Drive (safe no-op otherwise).
try:
    import google.colab  # type: ignore
    from google.colab import drive  # type: ignore
    if not os.path.exists('/content/drive'):
        os.makedirs('/content/drive', exist_ok=True)
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

MODEL_ID = os.environ.get('MODEL_ID', 'llava-hf/llava-1.5-7b-hf')


def has_stage_artifacts(work_dir):
    req = [
        f'{work_dir}/results/final_hallucination_heads.json',
        f'{work_dir}/cache/screening_state.pkl',
        f'{work_dir}/cache/selected_imgs.json',
    ]
    return all(os.path.exists(p) for p in req)


# 1) Use explicit WORK_DIR if provided.
# 2) Else, try common folders.
# 3) Else, scan one level under MyDrive for required artifacts.
# Accept either project root OR a direct .../results path.
work_dir_env = os.environ.get('WORK_DIR', '').strip()
candidates = []
if work_dir_env:
    env_path = Path(work_dir_env)
    candidates.append(str(env_path))
    if env_path.name == 'results':
        candidates.append(str(env_path.parent))

candidates.extend([
    '/content/drive/MyDrive/llava_hallucination_heads',
    '/content/drive/MyDrive/llava_hallucination_heads/results/..',
    '/content/drive/MyDrive/final project',
    '/content/drive/MyDrive/umass courses/spring26/682/final project',
])

mydrive = Path('/content/drive/MyDrive')
if mydrive.exists():
    for child in mydrive.iterdir():
        if child.is_dir():
            candidates.append(str(child))

seen = set()
ordered_candidates = []
for c in candidates:
    if c not in seen:
        ordered_candidates.append(c)
        seen.add(c)

WORK_DIR = None
for c in ordered_candidates:
    if has_stage_artifacts(c):
        WORK_DIR = c
        break

if WORK_DIR is None:
    checked = '\n'.join(f'  - {c}' for c in ordered_candidates[:25])
    raise FileNotFoundError(
        'Could not locate Stage 1/2 artifacts automatically.\n'
        'Set WORK_DIR manually to your project root OR to the results dir, e.g.\n'
        "os.environ['WORK_DIR'] = '/content/drive/MyDrive/llava_hallucination_heads'\n"
        "os.environ['WORK_DIR'] = '/content/drive/MyDrive/llava_hallucination_heads/results'\n"
        'Expected files under WORK_DIR:\n'
        '  - results/final_hallucination_heads.json\n'
        '  - cache/screening_state.pkl\n'
        '  - cache/selected_imgs.json\n\n'
        f'Checked candidates:\n{checked}'
    )

COCO_DIR = f'{WORK_DIR}/coco'
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)

HEADS_JSON = f'{WORK_DIR}/results/final_hallucination_heads.json'
CACHE_FILE = f'{WORK_DIR}/cache/screening_state.pkl'
SELECTED_IMGS_JSON = f'{WORK_DIR}/cache/selected_imgs.json'

results_dir = Path(f'{WORK_DIR}/results')

# Prefer explicit adapter path. If missing, auto-pick a matching adapter dir.
adapter_env = os.environ.get('STAGE4_LORA_ADAPTER_DIR', '').strip()
if adapter_env:
    STAGE4_LORA_ADAPTER_DIR = adapter_env
else:
    default_adapter = results_dir / 'stage2_lora_adapter'
    if default_adapter.exists():
        STAGE4_LORA_ADAPTER_DIR = str(default_adapter)
    else:
        tagged = sorted(results_dir.glob('stage2_lora_adapter*'))
        if not tagged:
            raise FileNotFoundError(
                f'No LoRA adapter found in {results_dir}. '
                'Set STAGE4_LORA_ADAPTER_DIR explicitly.'
            )
        STAGE4_LORA_ADAPTER_DIR = str(tagged[-1])

# Pick Stage 2 LoRA eval JSON for apples-to-apples comparison.
stage2_eval_env = os.environ.get('STAGE2_LORA_EVAL_PATH', '').strip()
if stage2_eval_env:
    STAGE2_LORA_EVAL_PATH = stage2_eval_env
else:
    default_eval = results_dir / 'stage2_lora_eval.json'
    if default_eval.exists():
        STAGE2_LORA_EVAL_PATH = str(default_eval)
    else:
        eval_candidates = sorted(results_dir.glob('stage2_lora_eval*.json'))
        STAGE2_LORA_EVAL_PATH = str(eval_candidates[-1]) if eval_candidates else ''


def _run_suffix(name, prefix):
    # Example: stage2_lora_adapter_abc -> _abc ; stage2_lora_eval_abc.json -> _abc
    if not name.startswith(prefix):
        return ''
    s = name[len(prefix):]
    if s.endswith('.json'):
        s = s[:-5]
    return s

def resolve_adapter_dir(path_str):
    """Resolve adapter dir even if user points to results/ parent folder."""
    p = Path(path_str)
    if not p.exists():
        return path_str

    if p.is_file():
        p = p.parent

    def is_adapter_dir(d):
        return (d / 'adapter_config.json').exists() and (
            (d / 'adapter_model.safetensors').exists() or (d / 'adapter_model.bin').exists()
        )

    if is_adapter_dir(p):
        return str(p)

    # If path is .../results or project root, search common locations.
    search_roots = [p]
    if (p / 'results').exists():
        search_roots.append(p / 'results')

    for root in search_roots:
        direct = root / 'stage2_lora_adapter'
        if is_adapter_dir(direct):
            return str(direct)

        tagged = sorted(root.glob('stage2_lora_adapter*'))
        for cand in reversed(tagged):
            if cand.is_dir() and is_adapter_dir(cand):
                return str(cand)

    return path_str


STAGE4_LORA_ADAPTER_DIR = resolve_adapter_dir(STAGE4_LORA_ADAPTER_DIR)

adapter_name = Path(STAGE4_LORA_ADAPTER_DIR).name
adapter_suffix = _run_suffix(adapter_name, 'stage2_lora_adapter')

eval_suffix = ''
if STAGE2_LORA_EVAL_PATH:
    eval_name = Path(STAGE2_LORA_EVAL_PATH).name
    eval_suffix = _run_suffix(eval_name, 'stage2_lora_eval')

if eval_suffix and adapter_suffix and (eval_suffix != adapter_suffix):
    print('[WARN] Adapter/eval run-tag mismatch detected:')
    print('  Adapter suffix:', adapter_suffix)
    print('  Stage2 eval suffix:', eval_suffix)
    print('  This can cause misleading Stage2 vs Stage4 comparisons.')

STAGE4_OUTPUT_TAG = os.environ.get('STAGE4_OUTPUT_TAG', '').strip()

THETA = float(os.environ.get('STAGE4_THETA', '0.08'))
ALPHA = float(os.environ.get('STAGE4_ALPHA', '8.0'))
MAX_NEW_TOKENS = int(os.environ.get('STAGE4_MAX_NEW_TOKENS', '80'))
EVAL_N = int(os.environ.get('STAGE4_EVAL_N', '40'))

PROMPT_TEMPLATE = 'USER: <image>\nDescribe the image in one sentence.\nASSISTANT:'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

for p in [HEADS_JSON, CACHE_FILE, SELECTED_IMGS_JSON, STAGE4_LORA_ADAPTER_DIR]:
    assert os.path.exists(p), f'Missing required path: {p}'

adapter_cfg = Path(STAGE4_LORA_ADAPTER_DIR) / 'adapter_config.json'
adapter_weights_ok = (
    (Path(STAGE4_LORA_ADAPTER_DIR) / 'adapter_model.safetensors').exists()
    or (Path(STAGE4_LORA_ADAPTER_DIR) / 'adapter_model.bin').exists()
)
assert adapter_cfg.exists(), f'Adapter config not found: {adapter_cfg}'
assert adapter_weights_ok, f'Adapter weights not found in: {STAGE4_LORA_ADAPTER_DIR}'

print('WORK_DIR:', WORK_DIR)
print('Adapter:', STAGE4_LORA_ADAPTER_DIR)
if STAGE2_LORA_EVAL_PATH:
    print('Stage2 eval file:', STAGE2_LORA_EVAL_PATH)
print('THETA:', THETA, '| ALPHA:', ALPHA, '| EVAL_N:', EVAL_N)

Mounted at /content/drive
Device: cuda
GPU: NVIDIA A100-SXM4-40GB
WORK_DIR: /content/drive/MyDrive/llava_hallucination_heads
Adapter: /content/drive/MyDrive/llava_hallucination_heads/results/stage2_lora_adapter
Stage2 eval file: /content/drive/MyDrive/llava_hallucination_heads/results/stage2_lora_eval.json
THETA: 0.08 | ALPHA: 8.0 | EVAL_N: 40


In [ ]:
# Load base model + LoRA adapter
# Base load is exactly Stage 3 style; Stage 4 adds only adapter attachment.
import subprocess
import sys

processor = AutoProcessor.from_pretrained(MODEL_ID)
base_model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
    low_cpu_mem_usage=True,
    attn_implementation='eager',
    device_map={'': 0} if device == 'cuda' else None,
)


def load_lora_or_raise(base_model_obj, adapter_dir):
    return PeftModel.from_pretrained(base_model_obj, adapter_dir)


try:
    model = load_lora_or_raise(base_model, STAGE4_LORA_ADAPTER_DIR)
except Exception as e:
    msg = str(e)
    # Common Colab/runtime issue: old torchao package conflicts with newer transformers/peft.
    if 'incompatible version of torchao' in msg.lower():
        print('[WARN] Incompatible torchao detected. Removing torchao and retrying once...')
        _ = subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
        try:
            model = load_lora_or_raise(base_model, STAGE4_LORA_ADAPTER_DIR)
        except Exception as e2:
            raise RuntimeError(
                'Failed to load LoRA adapter after torchao fallback.\n'
                f'Adapter dir: {STAGE4_LORA_ADAPTER_DIR}\n'
                f'Base model: {MODEL_ID}\n'
                f'Original error: {type(e).__name__}: {e}\n'
                f'After fallback: {type(e2).__name__}: {e2}\n'
                'Try runtime restart, then rerun from top.'
            )
    else:
        raise RuntimeError(
            'Failed to load LoRA adapter with PeftModel.from_pretrained.\n'
            f'Adapter dir: {STAGE4_LORA_ADAPTER_DIR}\n'
            f'Base model: {MODEL_ID}\n'
            'Check that adapter was trained on the same base model and that '
            '`adapter_config.json` + adapter weights exist in this folder.\n'
            'You can override path with:\n'
            "os.environ['STAGE4_LORA_ADAPTER_DIR'] = '/content/drive/MyDrive/.../results/stage2_lora_adapter_<tag>'\n"
            f'Original error: {type(e).__name__}: {e}'
        )

model.eval()

print('Loaded base + LoRA model for Stage 4 inference.')

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

[WARN] Incompatible torchao detected. Removing torchao and retrying once...
Loaded base + LoRA model for Stage 4 inference.


In [ ]:
# Rebuild Stage 2/3-compatible eval split from Stage 1 artifacts
with open(HEADS_JSON) as f:
    final_list = json.load(f)

risky_heads = defaultdict(set)
for h in final_list:
    risky_heads[int(h['layer'])].add(int(h['head']))

with open(CACHE_FILE, 'rb') as f:
    stage1_state = pickle.load(f)
_ = stage1_state.get('per_image_cache', {})  # kept for compatibility

with open(SELECTED_IMGS_JSON) as f:
    img_meta = json.load(f)

selected_imgs = img_meta['ids']
img_to_gt_objects = {int(k): set(v) for k, v in img_meta['gt_objects'].items()}

ann_path = f'{COCO_DIR}/annotations/instances_val2014.json'
cap_ann_path = f'{COCO_DIR}/annotations/captions_val2014.json'
img_dir = f'{COCO_DIR}/val2014_subset'

coco = COCO(ann_path)
_ = COCO(cap_ann_path)

img_meta_coco = coco.loadImgs(selected_imgs)
img_id_to_path = {m['id']: f"{img_dir}/{m['file_name']}" for m in img_meta_coco}

cat_id_to_name = {c['id']: c['name'].lower() for c in coco.loadCats(coco.getCatIds())}
ALL_COCO_OBJECTS = set(cat_id_to_name.values())

random.seed(42)
_eval_pool = [img_id for img_id in selected_imgs if img_id in img_id_to_path and os.path.exists(img_id_to_path[img_id])]
eval_images = _eval_pool[-EVAL_N:]
eval_gt_objects = [img_to_gt_objects[img_id] for img_id in eval_images]

print('Risky heads:', sum(len(v) for v in risky_heads.values()))
print('Eval images:', len(eval_images))
print('First 5 eval image IDs:', eval_images[:5])

loading annotations into memory...
Done (t=6.40s)
creating index...
index created!
loading annotations into memory...
Done (t=1.09s)
creating index...
index created!
Risky heads: 32
Eval images: 40
First 5 eval image IDs: [183973, 431479, 220638, 466766, 456261]


In [ ]:
# CHAIR helpers (mirrors Stage 1/2/3 vocabulary logic)
COCO_SYNONYMS = {
    'person': ['man','woman','people','boy','girl','child','guy','lady','kid','baby','player','rider','skier','surfer','snowboarder'],
    'car': ['vehicle','automobile','sedan','suv'],
    'dog': ['puppy','dogs'],
    'cat': ['kitten','cats'],
    'tv': ['television','monitor','screen'],
    'couch': ['sofa'],
    'cell phone': ['phone','cellphone','smartphone'],
    'dining table': ['table','desk'],
    'wine glass': ['glass'],
    'bicycle': ['bike'],
    'motorcycle': ['motorbike'],
    'airplane': ['plane','jet'],
    'potted plant': ['plant'],
    'laptop': ['computer'],
    'refrigerator': ['fridge'],
    'truck': ['lorry'],
    'boat': ['ship','sailboat'],
    'fire hydrant': ['hydrant'],
    'hot dog': ['hotdog'],
    'traffic light': ['stoplight'],
    'sports ball': ['ball','football','soccer ball','basketball'],
    'baseball bat': ['bat'],
    'tennis racket': ['racket','racquet'],
}
MULTIWORD_ALIASES = {
    'hydrant': 'fire hydrant',
    'hotdog': 'hot dog',
    'stoplight': 'traffic light',
    'bat': 'baseball bat',
    'racket': 'tennis racket',
    'racquet': 'tennis racket',
}

OBJECT_VOCAB = set(ALL_COCO_OBJECTS)
for syns in COCO_SYNONYMS.values():
    OBJECT_VOCAB.update(syns)
OBJECT_VOCAB.update(MULTIWORD_ALIASES.keys())


def normalize_obj_word(w):
    w = w.lower().strip()
    if w in MULTIWORD_ALIASES:
        w = MULTIWORD_ALIASES[w]
    return w


def find_caption_object_words(caption, gt_objects):
    gt_norm = {normalize_obj_word(x) for x in gt_objects}
    expanded_gt = set(gt_norm)
    for canonical, syns in COCO_SYNONYMS.items():
        if canonical in gt_norm:
            expanded_gt.update(syns)

    doc = nlp(caption)
    obj_words, hall_words = [], []
    for tok in doc:
        w = tok.text.lower().strip()
        if tok.pos_ not in ('NOUN', 'PROPN') or len(w) < 2:
            continue
        wn = normalize_obj_word(w)
        if wn in OBJECT_VOCAB:
            obj_words.append(wn)
            if wn not in expanded_gt:
                hall_words.append(wn)
    return obj_words, hall_words


def chair_eval_from_caption_fn(caption_fn, images, gt_objects_list):
    chairs_list, chairi_list = [], []
    examples = []
    for img_id, gt_set in tqdm(list(zip(images, gt_objects_list)), total=len(images), desc='CHAIR'):
        cap = caption_fn(img_id)
        obj_words, hall_words = find_caption_object_words(cap, gt_set)
        chairs_list.append(1 if len(hall_words) > 0 else 0)
        chairi_list.append(len(hall_words) / max(len(obj_words), 1))
        if len(examples) < 5:
            examples.append({'img_id': int(img_id), 'caption': cap, 'hall_words': hall_words})

    return {
        'CHAIRs': float(np.mean(chairs_list)) if chairs_list else 0.0,
        'CHAIRi': float(np.mean(chairi_list)) if chairi_list else 0.0,
        'n': len(chairs_list),
        'examples': examples,
    }

In [ ]:
# Build penalty token set + Stage 4 decoding functions

def object_token_ids(tokenizer, object_vocab):
    ids = set()
    for w in sorted(object_vocab):
        for form in [w, ' ' + w]:
            toks = tokenizer.encode(form, add_special_tokens=False)
            if len(toks) == 1:
                ids.add(int(toks[0]))
    return sorted(ids)

PENALTY_TOKEN_IDS = object_token_ids(processor.tokenizer, OBJECT_VOCAB)
print('Penalty token ids:', len(PENALTY_TOKEN_IDS))


@torch.no_grad()
def infer_visual_token_span(model_obj, image_path):
    img = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT_TEMPLATE, images=img, return_tensors='pt').to(device, torch.float16)
    inputs['input_ids'] = inputs['input_ids'].long()
    inputs['attention_mask'] = inputs['attention_mask'].long()

    out = model_obj(**inputs, output_attentions=True, return_dict=True)
    attn_len = out.attentions[0].shape[-1]
    text_len = inputs['input_ids'].shape[1]

    n_visual = max(0, attn_len - text_len)
    return 0, n_visual


def grounding_score_from_attentions(attentions, risky_map, visual_start, visual_end, eps=1e-8):
    if visual_end <= visual_start:
        return 0.0

    scores = []
    for layer_idx, heads in risky_map.items():
        if layer_idx >= len(attentions):
            continue

        layer_attn = attentions[layer_idx]  # [B, H, Q, K]
        if layer_attn is None:
            continue

        for h in heads:
            if h >= layer_attn.shape[1]:
                continue

            probs = layer_attn[0, h, -1, :]
            vis = probs[visual_start:visual_end]
            vis_mass = vis.sum()

            vis_norm = vis / (vis.sum() + eps)
            vis_entropy = -(vis_norm * (vis_norm + eps).log()).sum()

            score = (vis_mass / (vis_entropy + eps)).item()
            scores.append(score)

    return float(np.mean(scores)) if scores else 0.0


@torch.no_grad()
def generate_caption_baseline(model_obj, image_path, max_new_tokens=80):
    img = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT_TEMPLATE, images=img, return_tensors='pt').to(device, torch.float16)
    inputs['input_ids'] = inputs['input_ids'].long()
    inputs['attention_mask'] = inputs['attention_mask'].long()

    out = model_obj.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_dict_in_generate=True,
        use_cache=True,
    )
    gen_ids = out.sequences[0, inputs['input_ids'].shape[1]:]
    return processor.tokenizer.decode(gen_ids, skip_special_tokens=True)


@torch.no_grad()
def generate_caption_with_grounding_control(
    model_obj,
    image_path,
    theta=0.08,
    alpha=8.0,
    max_new_tokens=80,
    eos_token_id=None,
):
    img = Image.open(image_path).convert('RGB')
    inputs = processor(text=PROMPT_TEMPLATE, images=img, return_tensors='pt').to(device, torch.float16)
    input_ids = inputs['input_ids'].long()
    attention_mask = inputs['attention_mask'].long()
    pixel_values = inputs['pixel_values']

    visual_start, visual_end = infer_visual_token_span(model_obj, image_path)

    past_key_values = None
    generated = []

    for _ in range(max_new_tokens):
        if past_key_values is None:
            out = model_obj(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                use_cache=True,
                past_key_values=None,
                output_attentions=True,
                return_dict=True,
            )
        else:
            out = model_obj(
                input_ids=input_ids[:, -1:],
                attention_mask=attention_mask,
                use_cache=True,
                past_key_values=past_key_values,
                output_attentions=True,
                return_dict=True,
            )

        logits = out.logits[:, -1, :]
        g = grounding_score_from_attentions(out.attentions, risky_heads, visual_start, visual_end)

        if g < theta and PENALTY_TOKEN_IDS:
            penalty = alpha * (g - theta)  # negative when below threshold
            logits[:, PENALTY_TOKEN_IDS] += penalty

        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        token_id = int(next_token.item())
        generated.append(token_id)

        if eos_token_id is not None and token_id == eos_token_id:
            break

        if past_key_values is None:
            input_ids = torch.cat([input_ids, next_token], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)
        else:
            input_ids = next_token
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=1)

        past_key_values = out.past_key_values

    return processor.tokenizer.decode(generated, skip_special_tokens=True)

Penalty token ids: 55


In [ ]:
# Stage 4 evaluation: LoRA-only decoding vs LoRA+inference control

def caption_fn_lora_only(img_id):
    return generate_caption_baseline(model, img_id_to_path[img_id], max_new_tokens=MAX_NEW_TOKENS)


def caption_fn_lora_control(img_id):
    return generate_caption_with_grounding_control(
        model,
        img_id_to_path[img_id],
        theta=THETA,
        alpha=ALPHA,
        max_new_tokens=MAX_NEW_TOKENS,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

print('=== STAGE 4: LORA-ONLY CHAIR ===')
stage4_lora_only_chair = chair_eval_from_caption_fn(caption_fn_lora_only, eval_images, eval_gt_objects)
print({k: v for k, v in stage4_lora_only_chair.items() if k != 'examples'})

print('\n=== STAGE 4: LORA + INFERENCE CONTROL CHAIR ===')
stage4_lora_ctrl_chair = chair_eval_from_caption_fn(caption_fn_lora_control, eval_images, eval_gt_objects)
print({k: v for k, v in stage4_lora_ctrl_chair.items() if k != 'examples'})


def fmt_delta(after, before):
    return f'{after - before:+.4f}'

print('\n=== CHAIR DELTAS (lora+control - lora-only) ===')
print('CHAIRs:', fmt_delta(stage4_lora_ctrl_chair['CHAIRs'], stage4_lora_only_chair['CHAIRs']))
print('CHAIRi:', fmt_delta(stage4_lora_ctrl_chair['CHAIRi'], stage4_lora_only_chair['CHAIRi']))

=== STAGE 4: LORA-ONLY CHAIR ===


CHAIR:   0%|          | 0/40 [00:00<?, ?it/s]

{'CHAIRs': 0.15, 'CHAIRi': 0.04553571428571428, 'n': 40}

=== STAGE 4: LORA + INFERENCE CONTROL CHAIR ===


CHAIR:   0%|          | 0/40 [00:00<?, ?it/s]

{'CHAIRs': 0.125, 'CHAIRi': 0.049275362318840575, 'n': 40}

=== CHAIR DELTAS (lora+control - lora-only) ===
CHAIRs: -0.0250
CHAIRi: +0.0037


In [7]:
# Optional hyperparameter sweep for inference control (quick grid)
# Default grid from our discussion: theta in {0.06, 0.08, 0.10}, alpha in {4.0, 6.0, 8.0}
THETA_GRID = [float(x) for x in os.environ.get('STAGE4_THETA_GRID', '0.06,0.08,0.10').split(',')]
ALPHA_GRID = [float(x) for x in os.environ.get('STAGE4_ALPHA_GRID', '4.0,6.0,8.0').split(',')]

print('=== STAGE 4 GRID SEARCH (LoRA + control) ===')
print('Theta grid:', THETA_GRID)
print('Alpha grid:', ALPHA_GRID)

grid_results = []

for t in THETA_GRID:
    for a in ALPHA_GRID:
        print(f'\nRunning theta={t:.3f}, alpha={a:.3f}')

        def caption_fn_grid(img_id, _t=t, _a=a):
            return generate_caption_with_grounding_control(
                model,
                img_id_to_path[img_id],
                theta=_t,
                alpha=_a,
                max_new_tokens=MAX_NEW_TOKENS,
                eos_token_id=processor.tokenizer.eos_token_id,
            )

        res = chair_eval_from_caption_fn(caption_fn_grid, eval_images, eval_gt_objects)
        row = {
            'theta': float(t),
            'alpha': float(a),
            'CHAIRs': float(res['CHAIRs']),
            'CHAIRi': float(res['CHAIRi']),
            'n': int(res['n']),
        }
        grid_results.append(row)
        print({k: row[k] for k in ['theta', 'alpha', 'CHAIRs', 'CHAIRi', 'n']})

# Rank by CHAIRs first, then CHAIRi (both lower is better)
grid_results_sorted = sorted(grid_results, key=lambda x: (x['CHAIRs'], x['CHAIRi']))
best_grid = grid_results_sorted[0] if grid_results_sorted else None

print('\n=== GRID SEARCH RANKING (best first) ===')
for r in grid_results_sorted:
    print(
        f"theta={r['theta']:.3f}, alpha={r['alpha']:.3f} -> "
        f"CHAIRs={r['CHAIRs']:.4f}, CHAIRi={r['CHAIRi']:.4f}, n={r['n']}"
    )

if best_grid is not None:
    print('\n=== BEST GRID CONFIG ===')
    print(best_grid)

# Optional: persist sweep results
save_grid = os.environ.get('STAGE4_SAVE_GRID', '1').strip().lower() in {'1', 'true', 'yes'}
if save_grid:
    grid_suffix = f'_{STAGE4_OUTPUT_TAG}' if STAGE4_OUTPUT_TAG else ''
    grid_out = f'{WORK_DIR}/results/stage4_grid_search{grid_suffix}.json'
    with open(grid_out, 'w') as f:
        json.dump({'theta_grid': THETA_GRID, 'alpha_grid': ALPHA_GRID, 'results': grid_results_sorted}, f, indent=2)
    print('Saved grid search:', grid_out)

=== STAGE 4 GRID SEARCH (LoRA + control) ===
Theta grid: [0.06, 0.08, 0.1]
Alpha grid: [4.0, 6.0, 8.0]

Running theta=0.060, alpha=4.000


CHAIR:   0%|          | 0/40 [00:00<?, ?it/s]

{'theta': 0.06, 'alpha': 4.0, 'CHAIRs': 0.1, 'CHAIRi': 0.041666666666666664, 'n': 40}

Running theta=0.060, alpha=6.000


CHAIR:   0%|          | 0/40 [00:00<?, ?it/s]

{'theta': 0.06, 'alpha': 6.0, 'CHAIRs': 0.15, 'CHAIRi': 0.049275362318840575, 'n': 40}

Running theta=0.060, alpha=8.000


...

{'theta': 0.1, 'alpha': 4.0, 'CHAIRs': 0.15, 'CHAIRi': 0.049275362318840575, 'n': 40}

Running theta=0.100, alpha=6.000


CHAIR:   0%|          | 0/40 [00:00<?, ?it/s]

{'theta': 0.1, 'alpha': 6.0, 'CHAIRs': 0.125, 'CHAIRi': 0.045108695652173916, 'n': 40}

Running theta=0.100, alpha=8.000


CHAIR:   0%|          | 0/40 [00:00<?, ?it/s]

{'theta': 0.1, 'alpha': 8.0, 'CHAIRs': 0.125, 'CHAIRi': 0.049275362318840575, 'n': 40}

=== GRID SEARCH RANKING (best first) ===
theta=0.060, alpha=4.000 -> CHAIRs=0.1000, CHAIRi=0.0417, n=40
theta=0.100, alpha=6.000 -> CHAIRs=0.1250, CHAIRi=0.0451, n=40
theta=0.080, alpha=4.000 -> CHAIRs=0.1250, CHAIRi=0.0493, n=40
theta=0.080, alpha=8.000 -> CHAIRs=0.1250, CHAIRi=0.0493, n=40
theta=0.100, alpha=8.000 -> CHAIRs=0.1250, CHAIRi=0.0493, n=40
theta=0.060, alpha=6.000 -> CHAIRs=0.1500, CHAIRi=0.0493, n=40
theta=0.060, alpha=8.000 -> CHAIRs=0.1500, CHAIRi=0.0493, n=40
theta=0.080, alpha=6.000 -> CHAIRs=0.1500, CHAIRi=0.0493, n=40
theta=0.100, alpha=4.000 -> CHAIRs=0.1500, CHAIRi=0.0493, n=40

=== BEST GRID CONFIG ===
{'theta': 0.06, 'alpha': 4.0, 'CHAIRs': 0.1, 'CHAIRi': 0.041666666666666664, 'n': 40}
Saved grid search: /content/drive/MyDrive/llava_hallucination_heads/results/stage4_grid_search.json


In [8]:
# Save Stage 4 results JSON
suffix = f'_{STAGE4_OUTPUT_TAG}' if STAGE4_OUTPUT_TAG else ''
out_path = f'{WORK_DIR}/results/stage4_lora_inference_eval{suffix}.json'

stage4_results = {
    'config': {
        'theta': THETA,
        'alpha': ALPHA,
        'max_new_tokens': MAX_NEW_TOKENS,
        'eval_n': len(eval_images),
        'model_id': MODEL_ID,
        'lora_adapter_dir': STAGE4_LORA_ADAPTER_DIR,
    },
    'lora_only_chair': stage4_lora_only_chair,
    'lora_inference_control_chair': stage4_lora_ctrl_chair,
}

with open(out_path, 'w') as f:
    json.dump(stage4_results, f, indent=2)

print('Saved:', out_path)

# Optional quick comparison to Stage 2 LoRA result file
if STAGE2_LORA_EVAL_PATH and os.path.exists(STAGE2_LORA_EVAL_PATH):
    with open(STAGE2_LORA_EVAL_PATH) as f:
        stage2_lora_eval = json.load(f)
    if 'chair' in stage2_lora_eval:
        print('\nStage 2 LoRA CHAIR:', stage2_lora_eval['chair'])
        print('Stage 4 LoRA-only CHAIR:', {k: v for k, v in stage4_lora_only_chair.items() if k != 'examples'})
        print('Stage 4 LoRA+control CHAIR:', {k: v for k, v in stage4_lora_ctrl_chair.items() if k != 'examples'})
else:
    print('\nNo Stage 2 LoRA eval file found for comparison.')

Saved: /content/drive/MyDrive/llava_hallucination_heads/results/stage4_lora_inference_eval.json

Stage 2 LoRA CHAIR: {'CHAIRs': 0.0, 'CHAIRi': 0.0, 'n': 40}
Stage 4 LoRA-only CHAIR: {'CHAIRs': 0.15, 'CHAIRi': 0.04553571428571428, 'n': 40}
Stage 4 LoRA+control CHAIR: {'CHAIRs': 0.125, 'CHAIRi': 0.049275362318840575, 'n': 40}
